# Uno strato: la cosa di cui il modello e' fatto quasi per intero

*Anatomia di un LLM* · capitolo 3, «I mattoni» · nel libro: §3.4 «Uno strato» · script `01-03`, `scripts/01_03_neurone_e_strato.py`

**Dove siamo.** Lo strato è il mattone di cui il modello è fatto quasi per intero: proiezioni dell'attenzione, rete interna, testa di uscita. Qui si costruisce un neurone a mano, poi uno strato, e si controlla che siano la stessa cosa.

Un neurone somma un prodotto scalare e un numero. Uno strato e' una fila di
neuroni che guardano lo stesso ingresso. Il modello di questo libro e' fatto
quasi tutto di strati: le tre proiezioni dell'attenzione sono strati, la rete
del blocco sono due strati, la testa di uscita e' uno strato.

Si esegue con `python3 scripts/01_03_neurone_e_strato.py`
e dura circa 5 secondi.

> **Per riottenere i numeri del libro** esegui le celle in ordine, dalla prima. Il seme si fissa una volta sola, in `prepara()`: rieseguire una cella a metà cambia i numeri casuali di tutte quelle dopo.

[Apri in Colab](https://colab.research.google.com/github/fcollova/llm-anatomy/blob/main/notebooks/01_03_neurone_e_strato.ipynb) · la prima cella scarica il codice, se non lo trova accanto.

In [ ]:
# Porta il notebook nella cartella che contiene llm_anatomy/, da dovunque
# sia stato aperto: gli script si eseguono da li', e cosi' anche le celle.
import os
import subprocess
import sys
from pathlib import Path

radice = next((p for p in [Path.cwd(), *Path.cwd().parents]
               if (p / "llm_anatomy").is_dir()), None)

if radice is None:
    # Il notebook e' stato aperto da solo, per esempio su Colab: il codice, il
    # testo e il modello addestrato si scaricano una volta, qui accanto.
    REPO, RAMO, CARTELLA = "fcollova/llm-anatomy", "main", ""
    copia = Path("anatomia-di-un-llm")
    if not copia.exists():
        esito = subprocess.run(
            ["git", "clone", "-q", "--depth", "1", "--branch", RAMO,
             f"https://github.com/{REPO}.git", str(copia)], capture_output=True)
        if esito.returncode != 0:
            raise SystemExit("Non riesco a scaricare il codice da github.com/"
                             + REPO + ": " + esito.stderr.decode()[-300:])
    radice = copia / CARTELLA

os.chdir(radice)
sys.path.insert(0, str(Path.cwd()))

In [ ]:
import torch
import torch.nn as nn

from llm_anatomy.utils import (conta_parametri, prepara, stampa_shape, titolo,
                               verifica)

prepara()

d_in = 4     # quanti numeri entrano
d_out = 3    # quanti neuroni, quindi quanti numeri escono
B = 5        # quante righe si elaborano insieme

[ambiente] torch 2.13.0+cpu  device=cpu  threads=8  seed=1337


### Un neurone, a mano

> **Nel libro, §3.4 «Uno strato».** Qual è il pezzo che il modello ripete ovunque, e che cosa fa?

**Cosa guardare.** `y = w.x + b`: un prodotto scalare più un numero fisso, nient'altro.

In [ ]:
# Un neurone fa due cose: un prodotto scalare fra il suo vettore di pesi e
# l'ingresso, piu' un numero fisso. Nient'altro.
titolo("Un neurone, a mano")
x = torch.randn(d_in)
w = torch.randn(d_in)
b = torch.tensor(0.5)
y = torch.dot(w, x) + b
print(f"  x  {[round(v, 3) for v in x.tolist()]}")
print(f"  w  {[round(v, 3) for v in w.tolist()]}")
print(f"  b  {b.item():.3f}")
print(f"  y = w.x + b = {y.item():.4f}")


Un neurone, a mano
------------------
  x  [-2.026, -2.066, -1.205, -0.912]
  w  [-1.25, 0.803, -0.207, 0.054]
  b  0.500
  y = w.x + b = 1.5739


### Uno strato: molti neuroni sullo stesso ingresso

**Cosa guardare.** La riga dei parametri: 15 = 3×4 pesi + 3 numeri fissi. La forma di `weight` è `(uscite, ingressi)`.

**Prova a cambiare:** in cima al notebook porta `d_out` da 3 a 4, e riesegui dall'inizio.

<details><summary>Che cosa succede</summary>

I parametri diventano 20 = 4×4 + 4, e l'uscita `Y` passa da `(5, 3)` a `(5, 4)`. Tutte le verifiche successive passano ancora.

</details>

In [ ]:
titolo("Uno strato: molti neuroni sullo stesso ingresso")
strato = nn.Linear(d_in, d_out, bias=True)
stampa_shape("weight", strato.weight)
stampa_shape("bias", strato.bias)
print(f"  parametri: {conta_parametri(strato)} = "
      f"{d_out}*{d_in} pesi + {d_out} bias")

X = torch.randn(B, d_in)
Y = strato(X)
stampa_shape("X (B, d_in)", X)
stampa_shape("Y (B, d_out)", Y)


Uno strato: molti neuroni sullo stesso ingresso
-----------------------------------------------
  weight                       (3, 4)                 float32
  bias                         (3,)                   float32
  parametri: 15 = 3*4 pesi + 3 bias
  X (B, d_in)                  (5, 4)                 float32
  Y (B, d_out)                 (5, 3)                 float32


### È davvero un prodotto scalare per neurone

**Cosa guardare.** Le due verifiche: neurone per neurone, e in forma di matrice, danno lo stesso risultato di `nn.Linear`.

In [ ]:
titolo("E' davvero un prodotto scalare per neurone")
a_mano = torch.stack([
    torch.stack([torch.dot(strato.weight[j], X[i]) + strato.bias[j]
                 for j in range(d_out)])
    for i in range(B)
])
verifica(torch.allclose(a_mano, Y, atol=1e-6),
         "il calcolo neurone per neurone coincide con nn.Linear")
verifica(torch.allclose(X @ strato.weight.T + strato.bias, Y, atol=1e-6),
         "e coincide con la forma matriciale X W^T + b")


E' davvero un prodotto scalare per neurone
------------------------------------------
  [verifica superata] il calcolo neurone per neurone coincide con nn.Linear
  [verifica superata] e coincide con la forma matriciale X W^T + b


### Il più delle volte, oggi, il bias non c'è

**Cosa guardare.** La differenza è solo il termine fisso: 15 contro 12.

In [ ]:
titolo("Il piu' delle volte, oggi, il bias non c'e'")
senza = nn.Linear(d_in, d_out, bias=False)
print(f"  con bias:   {conta_parametri(strato)} parametri")
print(f"  senza bias: {conta_parametri(senza)} parametri")
print("  su d=4096 il risparmio e' trascurabile: si toglie perche' non serve,")
print("  non per risparmiare memoria")


Il piu' delle volte, oggi, il bias non c'e'
-------------------------------------------
  con bias:   15 parametri
  senza bias: 12 parametri
  su d=4096 il risparmio e' trascurabile: si toglie perche' non serve,
  non per risparmiare memoria
